# UGOT Robot Control Notebook

This notebook demonstrates how to control the **UGOT mecanum wheel vehicle** using Python. It covers:

1. **Setup** — connecting to the robot and loading vision models
2. **Movement reference** — driving, strafing, turning, and stopping
3. **Color detection** — reacting to recognized colors in real time
4. **AprilTag pick-and-place** — autonomously locating a tagged object, approaching it, and picking it up
5. **Word/text recognition** — reading text via the robot's camera


### Prerequisites
- The robot must be **powered on** and connected to the same Wi-Fi network/hotspot as the laptop.

## 1. Setup — Connect to the UGOT

Import the UGOT SDK and helper libraries, then:
- **Initialize** the UGOT at its IP address (opens a gRPC connection on port 50051).
- **Load vision models** that will be used later in the notebook.

Available models loaded here:
| Model | Purpose |
|---|---|
| `color_recognition` | Identifies dominant colors in the camera frame |
| `word_recognition` | Reads printed text (OCR) |
| `line_recognition` | Detects lines for line-following tasks |
| `apriltag_qrcode` | Detects AprilTag markers and QR codes |

In [ ]:
pip install torch torchvision ultralytics-thop opencv-python

In [19]:
pip show torch

Name: torch
Version: 2.10.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /Users/bachnguyen/Desktop/National-Youth-Tech-Championship-2026/venv/lib/python3.13/site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: torchvision, ultralytics, ultralytics-thop
Note: you may need to restart the kernel to use updated packages.


In [10]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize('192.168.88.1')  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models([
    'color_recognition',
    'word_recognition',
    'line_recognition',
    'apriltag_qrcode'
])

192.168.88.1:50051


True

In [ ]:
pip show torch

## 2. Movement API Reference

This cell is a **reference / scratch pad** for the core movement commands. None of these lines run as a sequence — uncomment and call them individually as needed.

### Key concepts
- **Mecanum wheels** allow the robot to move in any direction (strafe, diagonal) without rotating.
- `angle` is the direction of travel in degrees (0 = forward, 90 = right strafe, etc.).
- `speed` is in cm/s for movement and degrees/s for turning
- `times` combined with `unit` specifies duration: `unit=1` is cm for movement, `unit=2` is degrees of rotation for turns.
- `turn` direction: `2` = left, `3` = right.
- The **mechanical arm** has 3 joints (joint1 = base, joint2 = middle, joint3 = furthest).

In [12]:
# Timed straight movement
# Move forward (angle=0) at 30 cm/s for 70 cm.
dist = 50
spd = 30
#got.mecanum_translate_speed_times(angle=0, speed=spd, times=dist, unit=1)

# Continuous straight movement (runs until stopped)
# Strafe right (angle=90) at 20 cm/s speed indefinitely.
#got.mecanum_translate_speed_times(angle=90, speed=spd, times=dist, unit=1)


#got.mecanum_translate_speed_times(angle=180, speed=spd, times=dist, unit=1)

# Continuous straight movement (runs until stopped)
# Strafe right (angle=90) at 20 cm/s speed indefinitely.
#got.mecanum_translate_speed_times(angle=-90, speed=spd, times=dist, unit=1)
# Timed turn
# Turn right (turn=3) at 45 deg/s for 180 degrees.
#got.mecanum_turn_speed_times(turn=3, speed=45, times=180, unit=2)

#  Continuous turn (runs until stopped) 
# Turn left (turn=2) at 20 deg/s speed indefinitely.
#got.mecanum_turn_speed(turn=2, speed=20)  # turn: left=2, right=3

#  Stop all wheel motors immediately 
#got.mecanum_stop()


got.mechanical_clamp_release() # Open the gripper
time.sleep(2)
got.mechanical_clamp_close() # Close the gripper

#  Arm reset 
# Returns all arm joints to their default/home position.
got.mechanical_arms_restory()

#  Move all arm joints simultaneously 
# angle1/2/3 are target angles (degrees); duration is movement time in ms.
got.mechanical_joint_control(angle1=0, angle2=0, angle3=0, duration=500)

#  Move a single arm joint 
# joint: 1=base, 2=middle, 3=furthest; angle in degrees; duration in ms.
#got.mechanical_single_joint_control(joint=1, angle=0, duration=500)

## 3. Real-Time Color Detection

This loop continuously reads the dominant color from the camera and updates the robot's **screen background** to match:
- **Red detected** → red background (theme 3)
- **Green detected** → green background (theme 6)
- **Anything else** → black/default background (theme 0)

The current color is also printed to the console. `clear_output()` refreshes the output each iteration so only the latest value is shown.

> **Stop the loop** by pressing the **Stop** button next to the cell or using **Interrupt**.

In [44]:
try:
    red_seen = False
    green_seen = False

    while True:
        # get_color_total_info() returns a list of detected color results.
        # Index [0] gives the name of the most prominent color (e.g. "Red", "Green").
        color = got.get_color_total_info()[0]

        # Update the robot's display based on detected color.
        if not red_seen:
            if color == "Red":
                got.screen_display_background(3)
                red_seen = True
                print("Red Detected!")
                break
        else:

            if color == "Green":
                got.screen_display_background(6)
                green_seen = True
                print("Green Deteced!")
                break

        # Print current color, or a message if nothing is detected.

        # Clear console output so each refresh shows only the latest color.
        clear_output(wait=True)
    print("Seen both")
    time.sleep(1)
    got.screen_clear()

except KeyboardInterrupt:
    # Safely stop the robot's wheels if the cell is interrupted.
    got.mecanum_stop()
    print("Done")

Done


## 4. AprilTag-Based Pick-and-Place

These two helper functions work together to **locate and pick up an object** marked with an AprilTag:

### `AP_centralization_approaching()`
Centers the robot on the tag and drives toward it until it reaches the target distance.
- Compares the tag's **X position** in the camera frame (0–640 px, center = 320) against a `gap` tolerance.
- If the tag is off-center, the robot **strafes** left or right to re-align.
- Once aligned, it **drives forward** until the tag is within `distance` meters.

### `pick_up()`
Uses the tag's current position and distance to calculate arm joint angles and grab the object:
1. Opens the gripper and raises the arm slightly to a ready position.
2. Computes `joint1` (horizontal aim) from tag X offset and `joint3` (reach) from tag distance.
3. Moves the arm to the computed pose, closes the gripper, then lifts the arm back up.

> **Joint angle math:**
> - `joint1 = (AP_x - 320) * -1/10` — scales pixel offset to degrees; negative sign corrects mirror orientation.
> - `joint3 = AP_distance * 100 - 80` — scales distance (meters) to an elbow extension angle.

In [73]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        # Refresh tag data every iteration for responsive corrections.
        AP_info = got.get_apriltag_total_info()
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        if AP_x < 320 - gap:
            # Tag is to the LEFT of center — strafe left to re-align.
            # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            # Tag is to the RIGHT of center — strafe right to re-align.
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            # Tag is centered but still too far — drive straight forward.
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            # Tag is centered AND within target distance — stop and exit.
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1]
    AP_distance = AP_info[0][6]

    # Move arm to a neutral ready position and open the gripper.
    # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
    got.mechanical_joint_control(0, 30, 30, 1000)
    got.mechanical_clamp_release() # Open gripper before extending arm
    time.sleep(2) # Wait for gripper to fully open

    # Calculate arm joint angles based on the tag's camera position.
    # joint1 (base): convert pixel offset from center to degrees.
    #   Negative factor corrects for the camera being mirrored horizontally.
    joint1 = int((AP_x - 320) * -1 / 10)

    # joint3 (furthest): convert distance (m) to an extension angle.
    # The -80 offset accounts for the arm's resting angle calibration.
    joint3 = int(AP_distance * 100 - 80)

    # Move arm to the computed pick-up pose.
    got.mechanical_joint_control(joint1, 0, joint3, 500)
    print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
    time.sleep(1) # Wait for arm to reach the target pose

    # Grasp the object and lift the arm back to the carry position.
    got.mechanical_clamp_close()
    time.sleep(2)  # Wait for gripper to fully close before lifting
    got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose

### Run the Pick-and-Place Sequence

This cell watches for an AprilTag and, once one is detected, calls the two functions above in sequence:
1. Approach and center on the tag.
2. Pick up the tagged object.

The loop then breaks after a successful pick. Interrupt the cell at any time to stop safely.

In [21]:
try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # A tag is visible — approach it and pick up the object.
            # Lower speeds (fwd_spd=5, strafe_spd=7) for more precise final approach.
            AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=5, strafe_spd=7)
            pick_up()

            print("Picked up!")
            break  # Exit after one successful pick-and-place

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

Done


In [23]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y


In [24]:
cmds = ["left", "right", "right", "left"]

def turn_left():
    # Keep rotating left until the line is found and centered
    while True:
        got.mecanum_move_xyz(0, 0, -20)  # rotate left in place
        
        offset, line_type, x, y = got.get_single_track_total_info()
        print(f"offset={offset}, type={line_type}")
        
        if abs(offset) < 10:  # line is roughly centered again
            got.mecanum_stop()
            break
def turn_right():
    # Keep rotating left until the line is found and centered
    while True:
        got.mecanum_move_xyz(0, 0, 20)  # rotate left in place
        
        offset, line_type, x, y = got.get_single_track_total_info()
        print(f"offset={offset}, type={line_type}")
        
        if abs(offset) < 10:  # line is roughly centered again
            got.mecanum_stop()
            break

In [25]:
got.open_camera()


def display_frame():
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Reads in the image data
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Turns numbers into color
        frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)

        display(Image2.fromarray(frame_rgb))
        clear_output(wait=True)

In [13]:
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

Done!


In [29]:
moves = 0
try:
    while True:
        display_frame()
        got.set_track_recognition_line(line_type=0)
        line_type, x, y = line_follow(mult=0.25, speed=35)

        if line_type in [0, 2]: # 0 = no line detected, 2 = y-intersection detected
            if line_type == 0:
                print("Off the line")
                got.mecanum_stop()
                break
            if line_type == 2:
                print("Y Intersection")
                if(cmds[moves] == 'left'): turn_left()
                else: turn_right()
                moves += 1
            else:
                print("Move Straight")
            if moves > len(cmds): break
    print("Done")

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")

Off the line
Done


## 5. Real-Time Text (Word) Recognition

This loop uses the robot's **OCR (optical character recognition) model** to continuously read text visible in the camera frame and print the result.

`get_words_result()` returns a string of any detected text, or an empty string if nothing is readable.

In [87]:
try:
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # Refresh output each frame so only the latest result is shown.
        # wait=True prevents flickering by waiting for new output before clearing.
        clear_output(wait=True)

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

Done
